## 1. Mount Google Drive


In [8]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Skipping drive mount:', e)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Clone repo

Đang clone từ **`ChauDucToan/NLP_DHM`**

In [9]:
import os
REPO_URL = 'https://github.com/ChauDucToan/NLP_DHM.git'
REPO_DIR = '/content/NLP_DHM'
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only || true
%cd $REPO_DIR
!pwd && ls


Already up to date.
/content/NLP_DHM
/content/NLP_DHM
configs  LICENSE    pyproject.toml  requirements.txt  src
data	 notebooks  README.md	    scripts	      tests


## 3. Cài đặt dependencies cơ bản

Cài các package thuần Python (`torch`, `sentencepiece`, `sacrebleu`, ...). CUDA fast-path cho Mamba sẽ được cài ở ô **3b** ngay bên dưới.


In [10]:
!pip install -q -r requirements.txt
import torch, sys
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
sys.path.insert(0, 'src')


torch: 2.10.0+cu128 | CUDA: True | device: Tesla T4


### 3b. Tăng tốc Mamba bằng CUDA kernels (tuỳ chọn nhưng khuyên dùng)

Cài wheel **trực tiếp từ GitHub Releases** thay vì build từ PyPI (PyPI sẽ cố compile và hầu như fail trên Colab vì lệch ABI / CUDA toolkit). Mất ~30 giây. Train sẽ nhanh ~5–10× nếu thành công, fallback PyTorch vẫn chạy đúng nếu fail.


In [11]:
# === Optional but recommended: CUDA fast-path (5–10× speedup) ===
# `pip install mamba-ssm causal-conv1d` from PyPI tries to *build* from source
# on Colab and almost always fails (CUDA toolkit / torch ABI mismatch).
# Instead, install matching prebuilt wheels straight from the GitHub releases.
import sys, subprocess, torch

# Skip on CPU runtimes (no benefit, and the wheels need CUDA).
if not torch.cuda.is_available():
    print('No CUDA → skipping fast-path install (pure-PyTorch fallback will run).')
else:
    torch_mm  = '.'.join(torch.__version__.split('+')[0].split('.')[:2])  # e.g. '2.10'
    cuda_maj  = torch.version.cuda.split('.')[0]                          # e.g. '12'
    py_tag    = f'cp{sys.version_info.major}{sys.version_info.minor}'     # e.g. 'cp312'
    abi       = 'TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE'
    suffix    = f'cu{cuda_maj}torch{torch_mm}cxx11abi{abi}-{py_tag}-{py_tag}-linux_x86_64.whl'

    ccv1d_ver = '1.6.1'
    mamba_ver = '2.3.1'
    ccv1d_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v{ccv1d_ver}.post4/causal_conv1d-{ccv1d_ver}+{suffix}'
    mamba_url = f'https://github.com/state-spaces/mamba/releases/download/v{mamba_ver}/mamba_ssm-{mamba_ver}+{suffix}'

    print('Installing causal-conv1d:', ccv1d_url.rsplit("/", 1)[-1])
    r1 = subprocess.run(['pip', 'install', '-q', '--no-deps', ccv1d_url])
    print('Installing mamba-ssm:    ', mamba_url.rsplit("/", 1)[-1])
    r2 = subprocess.run(['pip', 'install', '-q', '--no-deps', mamba_url])

    ok = True
    try:
        import causal_conv1d, mamba_ssm  # noqa
        from mamba_ssm.ops.selective_scan_interface import selective_scan_fn  # noqa
        from causal_conv1d import causal_conv1d_fn  # noqa
    except Exception as e:
        ok = False
        print('FAST-PATH INSTALL FAILED:', e)
        print(' → falling back to pure-PyTorch Mamba (training still works, ~5–10× slower).')
    if ok:
        print('CUDA fast-path active: selective_scan_fn + causal_conv1d_fn imported OK.')


Installing causal-conv1d: causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Installing mamba-ssm:     mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
CUDA fast-path active: selective_scan_fn + causal_conv1d_fn imported OK.


## 4. Demo sản phẩm

In [20]:
import os
import torch
from bi_mamba_mt.model import BiMambaTranslator, ModelConfig
from bi_mamba_mt.tokenizer import Tokenizer
from bi_mamba_mt.translate import translate_batch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_candidates = [
    "/content/drive/MyDrive/bi_mamba_55m/final.pt",
]
ckpt_path = next(p for p in _candidates if os.path.exists(p))
print('Loading', ckpt_path)
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
model = BiMambaTranslator(ModelConfig(**ckpt['model_cfg'])).to(device)
model.load_state_dict(ckpt['model'], strict=False); model.eval()
tok = Tokenizer(f"/content/drive/MyDrive/bi_mamba_55m/tokenizer/spm.model")

# OpenCC Trad→Simp normalisation for zh inputs. The model is trained on
# Mandarin Simplified (after PR #8 cleanup), so any Traditional input from
# the user should be converted before translation. If `opencc` isn't
# installed (or the user already typed Simplified), `_norm_zh` is a no-op.
try:
    import opencc
    _OPENCC_T2S = opencc.OpenCC('t2s')
    print('OpenCC t2s loaded — zh inputs will be normalised to Simplified')
except ImportError:
    _OPENCC_T2S = None
    print('opencc not installed — zh inputs are passed through as-is')

def _norm_zh(s: str) -> str:
    return _OPENCC_T2S.convert(s) if _OPENCC_T2S is not None else s


Loading /content/drive/MyDrive/bi_mamba_55m/final.pt


### 4.1. Dịch từ Việt sang Trung

In [23]:
print('--- vi → zh ---')
while True:
    s = input("Bạn hãy nhập một câu để dịch: ")
    t = translate_batch(model, tok, [s], 'vi2zh', beam_size=4, device=device)
    print(f'  {s!r} -> {t[0]!r}')

--- vi → zh ---
Bạn hãy nhập một câu để dịch: Xin chào mọi người
  'Xin chào mọi người' -> '大家好'
Bạn hãy nhập một câu để dịch: Xử lý ngôn ngữ tự nhiên
  'Xử lý ngôn ngữ tự nhiên' -> '自然语言处理。'


KeyboardInterrupt: Interrupted by user

### 4.2. Dịch từ Trung sang Việt

In [24]:
print('--- zh → vi ---')
while True:
    s = input("Bạn hãy nhập một câu để dịch: ")
    s_norm = _norm_zh(s)
    if s_norm != s:
        print(f'  (normalised) {s!r} → {s_norm!r}')
    t = translate_batch(model, tok, [s_norm], 'zh2vi', beam_size=4, device=device)
    print(f'  {s_norm!r} -> {t[0]!r}')


--- zh → vi ---
Bạn hãy nhập một câu để dịch: 大家好
  '大家好' -> 'Xin chào mọi người'
Bạn hãy nhập một câu để dịch: 自然语言处理
  '自然语言处理' -> 'Ngôn ngữ tự nhiên'


KeyboardInterrupt: Interrupted by user